In [ ]:
%load_ext autoreload 
%autoreload 2

import os

import torch
from dotenv import load_dotenv
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
)

load_dotenv()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

token = os.environ["HF_TOKEN"]

# model_name = "deepseek-ai/deepseek-llm-7b-base"
model_name = "meta-llama/Llama-3.1-8B"
# model_name = "google/gemma-7b"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
    token=token,
)
# model.generation_config = GenerationConfig.from_pretrained(model_name)
# model.generation_config.pad_token_id = model.generation_config.eos_token_id


In [ ]:
# Query model
def query_model(prompt, max_new_tokens):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    output = model.generate(input_ids, max_new_tokens=max_new_tokens)
    response = tokenizer.decode(output[0], skip_special_tokens=True)

    return response


prompt = """
Sono un eccellente linguista, specializzato nel rintracciare entità in testi giuridici. Il mio compito è di etichettare usando BIO tagging tutte le entità nominate in un testo come segue:
- le persone sono etichettate tra doppie parentesi tonde: ((PER))
- i luoghi sono etichettati tra doppie parentesi quadre: [[LOC]]
- le organizzazioni sono etichettate tra doppie parentesi graffe: {{ORG}} 
- le misc sono etichettate tra doppie parentesi angolari: <<MISC>>

Ecco alcuni esempi:


Input: Del resto l’atteggiamento di colpevole condivisione della stra tegia stragistica voluta dal Luca Rinaldi anche per l’eliminazione del dr Paolo Conti era provata non solo dal ruolo preminente che Andrea Moretti aveva continuato a rivestire nell’organizzazione anche dopo l’attentato di Marina Alta, alla luce dalle dichiarazioni di tutti i colla boranti ancora in libertà all’epoca, ma anche dalla sua partecipazione all’incontro nel quale circa un mese dopo la strage il Luca Rinaldi aveva fe steggiato in casa del Francesco Villa con un brindisi, in sieme a Alessandro Greco, Riccardo Galli, Stefano Romano, Davide Marchetti ed appunto il Andrea Moretti, la riuscita del barbaro crimine.
Output: Del resto l’atteggiamento di colpevole condivisione della stra tegia stragistica voluta dal ((Luca Rinaldi)) anche per l’eliminazione del dr ((Paolo Conti)) era provata non solo dal ruolo preminente che ((Andrea Moretti)) aveva continuato a rivestire nell’organizzazione anche dopo l’attentato di [[Marina Alta]], alla luce dalle dichiarazioni di tutti i colla boranti ancora in libertà all’epoca, ma anche dalla sua partecipazione all’incontro nel quale circa un mese dopo la strage il ((Luca Rinaldi)) aveva fe steggiato in casa del ((Francesco Villa)) con un brindisi, in sieme a ((Alessandro Greco)), ((Riccardo Galli)), ((Stefano Romano)), ((Davide Marchetti)) ed appunto il ((Andrea Moretti)), la riuscita del barbaro crimine.

Input: MESSINA Leonardo ha riferito che la commissione regionale di Consorzio Civico Tirrenico era costituita tra i rappresentanti delle province di Castelvento, Rocca Serena, Pian del Lago, San Felice al Piano, Monteluce e Colle Azzurro per decidere sulle questioni più rilevanti che superavano l’ambito territoriale di ciascuna provincia, all’interno della quale operavano le singole commissioni provinciali nonché per guidare e coordinare le attività delle varie province.
Output: ((MESSINA Leonardo)) ha riferito che la commissione regionale di {{Consorzio Civico Tirrenico}} era costituita tra i rappresentanti delle province di [[Castelvento]], [[Rocca Serena]], [[Pian del Lago]], [[San Felice al Piano]], [[Monteluce]] e [[Colle Azzurro]] per decidere sulle questioni più rilevanti che superavano l’ambito territoriale di ciascuna provincia, all’interno della quale operavano le singole commissioni provinciali nonché per guidare e coordinare le attività delle varie province.

Input: successo uno dei giorni che si aspettava l'arrivo del Dottore Paolo Conti Matteo De Santis Matteo De Santis praticamente la mattina quando stava per venire all'appuntamento con tutti noi, lui era con la sua macchina, ha avuto un incidente stradale, questo incidente l’ha avuto nello svincolo di Pietramonte, perché si esce dal, dall'autostrada in direzione Pietramonte, Marina Alta, e poi si gira in direzione Marina Alta, proprio in questo punto, dove lui aveva lo stop, non si è fermato allo stop e ha avuto un incidente, lui aveva allora una, una Automobili Levante, era una Automobili Levante bianca, mi pare, io non sono molto bravo nelle macchine piccole, comunque dovrebbe essere una, era un modello 205, mi pare era il modello 205.
Output: successo uno dei giorni che si aspettava l'arrivo del Dottore ((Paolo Conti)) ((Matteo De Santis)) praticamente la mattina quando stava per venire all'appuntamento con tutti noi, lui era con la sua macchina, ha avuto un incidente stradale, questo incidente l’ha avuto nello svincolo di [[Pietramonte]], perché si esce dal, dall'autostrada in direzione [[Pietramonte]], [[Marina Alta]], e poi si gira in direzione [[Marina Alta]], proprio in questo punto, dove lui aveva lo stop, non si è fermato allo stop e ha avuto un incidente, lui aveva allora una, una Automobili Levante, era una Automobili Levante bianca, mi pare, io non sono molto bravo nelle macchine piccole, comunque dovrebbe essere una, era un modello 205, mi pare era il modello 205.

Input: Rivafosca e dalla documentazione acquisita si evince che: il Nicola Romano gestì nei primi anni '60 un distributore di carburanti unitamente a Andrea Moretti; la deliberazione della Comune di Valleverde con cui venne accolta, per la parte di competenza del Agenzia Regionale per i Servizi, la domanda tendente ad ottenere la concessione per l’impianto dei distributori di carburante da collocare sul suolo pubblico di Via del Mercato all’angolo con Piazza delle Magnolie, fu adottata il 9 dicembre 1961, mentre era Sindaco Antonio Ferraro (in una adunanza della Consiglio Direttivo presieduta dal Vice Sindaco avv. Dario Pellegrini, con la presenza degli assessori effettivi Enrico Fontana, Simone Costa, Giorgio Lombardi, Fabio Mancini, Michele Rizzo, e degli assessori supplenti Giulia Bianchi Giulia Bianchi, Roberto Caruso, Claudio Serra, come si evince dalla documentazione acquisita all’udienza del 22 maggio 1996).
Output: ((Rivafosca)) e dalla documentazione acquisita si evince che: il ((Nicola Romano)) gestì nei primi anni '60 un distributore di carburanti unitamente a ((Andrea Moretti)); la deliberazione della {{Comune di Valleverde}} con cui venne accolta, per la parte di competenza del {{Agenzia Regionale per i Servizi}}, la domanda tendente ad ottenere la concessione per l’impianto dei distributori di carburante da collocare sul suolo pubblico di Via del Mercato all’angolo con [[Piazza delle Magnolie]], fu adottata il 9 dicembre 1961, mentre era Sindaco ((Antonio Ferraro)) (in una adunanza della Consiglio Direttivo presieduta dal Vice Sindaco avv. ((Dario Pellegrini)), con la presenza degli assessori effettivi ((Enrico Fontana)), ((Simone Costa)), ((Giorgio Lombardi)), ((Fabio Mancini)), ((Michele Rizzo)), e degli assessori supplenti ((Giulia Bianchi Giulia Bianchi)), ((Roberto Caruso)), ((Claudio Serra)), come si evince dalla documentazione acquisita all’udienza del 22 maggio 1996).

"""


def create_query(text: str) -> str:
    sent = f"Input: {text}\nOutput:"

    return prompt + sent


query = create_query(
    "Il Riccardo Galli ha specificato di aver conosciuto il Paride Valli presso la macelleria, sita nel quartiere Salamini, dei propri cugini Ettore Vitale, Ottavio Neri ed Leandro Sala (l’altro collaborante già citato), frequentata dal padre Riccardo Galli e da altri uomini d’altri tempi (come Gambino Simone Costa) e che il Paride Valli era persona vicina ed amica di Tullio Conte Di Napoli, circostanza anche personalmente constatata dal collaboratore allorché, in alcune occasioni, aveva incontrato il Paride Valli in compagnia del Di Napoli nella villa di proprietà di Enrico Fontana Gino Basile, altro uomo d’altri tempi della famiglia di Orlando Pace, nonché parente dello stesso Paride Valli ove lo stesso Di Napoli dimorava durante la sua gita."
)

response = query_model(query, max_new_tokens=300)

print(response[len(prompt) :])

In [ ]:
response[len(prompt) :].split("\n\n")[0].split("\n")[1][8:]

In [ ]:
import json
from datetime import datetime

from tqdm import tqdm

# Load jsonl file into a list of dictionaries
with open("../data/labelled/test.jsonl") as test_f:
    test_data = [json.loads(line) for line in test_f]


output = []
for sent in tqdm(test_data):
    query = create_query(sent["text"])
    response = query_model(query, max_new_tokens=300)

    output.append(response[len(prompt) :].split("\n\n")[1][8:])

# Create timestamp to create unique output files
dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")

# Save predictions to file
output_file = f"../output/llm_ner_pred_{model_name.split('/')[-1]}_{dt}.txt"
with open(output_file, "w") as test_f:
    for line in output:
        test_f.writelines(line + "\n")

## Evaluation

### Span-based

In [ ]:
import json
import re

# pred_filename = "llm_ner_pred_gemma-7b_2025-04-29_12-46-43"
pred_filename = "llm_ner_pred_Llama-3.1-8B_2025-04-29_11-31-28"
# pred_filename = "llm_ner_pred_deepseek-llm-7b-base_2025-04-28_14-54-20"

tmp_output_file = f"../output/{pred_filename}.txt"

with open(tmp_output_file, "r") as f:
    output = [line.strip() for line in f.readlines()]

# For each line in output, extract the predicted entity, its tag and its start/end indexes using regular expressions
output_entities = []
pattern = r"\(\((.*?)\)\)"

patterns = [
    (r"\(\((.*?)\)\)", "PER"),
    (r"\[\[(.*?)\]\]", "LOC"),
    (r"\{\{(.*?)\}\}", "ORG"),
    (r"\<\<(.*?)\>\>", "MISC"),
]

delimiters = ["((", "))", "[[", "]]", r"{{", r"}}", "<<", ">>"]


def replace_all(old: str, to_replace: list, new: str):
    for text in to_replace:
        old = old.replace(text, new)

    return old


In [ ]:
pred = []
for i, line in enumerate(output):
    match = re.search(pattern, line)

    labels = []
    for pattern, label in patterns:
        for match in re.finditer(pattern, line):
            content = match.group(1)

            start_idx = match.start()
            end_idx = match.end()

            # Count number of double parentheses seen up to start_idx
            parentheses_count = (
                line[:end_idx].count("((")
                + line[:end_idx].count("))")
                + line[:end_idx].count("[[")
                + line[:end_idx].count("]]")
                + line[:end_idx].count(r"{{")
                + line[:end_idx].count(r"}}")
                + line[:end_idx].count(r"<<")
                + line[:end_idx].count(r">>")
            )

            start_idx = match.start() - (parentheses_count * 2) + 4
            end_idx = match.end() - (parentheses_count * 2)

            labels.append([start_idx, end_idx, label])

    pred.append(
        {
            "idx": i,
            "text": replace_all(line, delimiters, ""),
            "pred_text": line,
            "label": labels,
        }
    )

# Save as jsonl file
with open(tmp_output_file.replace(".txt", "_eval.json"), "w") as file:
    file.write(json.dumps(pred, ensure_ascii=False))

In [ ]:
import json

from util import evaluate_ner

with open(f"../output/{pred_filename}_eval.json") as file:
    pred = json.load(file)

with open("../data/labelled/test_combined.jsonl") as file:
    true = [json.loads(line) for line in file.readlines()]

pred_lines = [p["label"] for p in pred]
true_lines = [t["label"] for t in true]

# Append index to each label to ensure unique elements when lists have been flattened
for i, line in enumerate(true_lines):
    for label in line:
        label.append(i)

for i, line in enumerate(pred_lines):
    for label in line:
        label.append(i)

In [ ]:
from itertools import chain

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(pred_lines)))
flat_golds = list(map(tuple, chain.from_iterable(true_lines)))

evaluate_ner(flat_preds, flat_golds)